# 03 — Option Pricing: Black-Scholes, Monte Carlo, and Greeks

We price European options three ways: the closed-form Black-Scholes formula, Monte Carlo simulation under the risk-neutral measure (reusing our GBM machinery from Module 2), and compare convergence. Then we compute the Greeks both analytically and via finite differences, and visualize their shape across strikes and time to expiry.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quant_sims.options import (
    BlackScholesParams,
    call_price,
    put_price,
    put_call_parity_check,
    all_greeks,
    all_fd_greeks,
    monte_carlo_price,
    convergence_study,
    delta,
    gamma,
    vega,
    theta,
)

%matplotlib inline

## 1. Black-Scholes closed-form pricing

Standard example: S=K=$100, 1 year to expiry, 5% risk-free rate, 20% volatility.

In [ ]:
params = BlackScholesParams(S=100.0, K=100.0, T=1.0, r=0.05, sigma=0.2)

c = call_price(params)
p = put_price(params)
print(f"Call price: {c:.4f}")
print(f"Put price:  {p:.4f}")
print(f"Put-call parity holds: {put_call_parity_check(params)}")

## 2. Monte Carlo pricing and convergence to Black-Scholes

We simulate the underlying under the risk-neutral measure (drift = r, not the real-world mu), compute the discounted expected payoff, and show that the estimate converges to the closed-form price as the number of simulations grows -- with error shrinking at the classic Monte Carlo rate of ~1/sqrt(N).

In [ ]:
mc_est, mc_se = monte_carlo_price(params, option_type="call", n_simulations=500_000, seed=42, return_std_error=True)
print(f"Monte Carlo call price: {mc_est:.4f} +/- {1.96*mc_se:.4f} (95% CI)")
print(f"Black-Scholes call price: {c:.4f}")
print(f"Difference: {abs(mc_est - c):.4f}")

In [ ]:
result = convergence_study(params, option_type="call", seed=7)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(result["sample_sizes"], result["mc_prices"], "o-", color="steelblue", label="MC estimate")
axes[0].axhline(result["bs_price"], color="darkorange", linestyle="--", label=f"Black-Scholes = {result['bs_price']:.4f}")
axes[0].fill_between(result["sample_sizes"], result["mc_prices"] - 1.96*result["std_errors"], result["mc_prices"] + 1.96*result["std_errors"], alpha=0.2, color="steelblue")
axes[0].set_xscale("log")
axes[0].set_xlabel("Number of simulations")
axes[0].set_ylabel("Estimated call price")
axes[0].set_title("MC price convergence (with 95% CI)")
axes[0].legend()

axes[1].loglog(result["sample_sizes"], result["std_errors"], "o-", color="steelblue", label="MC standard error")
reference_slope = result["std_errors"][0] / np.sqrt(result["sample_sizes"] / result["sample_sizes"][0])
axes[1].loglog(result["sample_sizes"], reference_slope, "--", color="gray", label="1/sqrt(N) reference")
axes[1].set_xlabel("Number of simulations")
axes[1].set_ylabel("Standard error")
axes[1].set_title("MC error shrinks at the classic 1/sqrt(N) rate")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Greeks: analytical vs. finite-difference

We compute the Greeks two ways and confirm they agree. In practice, finite differences are the fallback when no closed form exists (e.g. exotic payoffs, path-dependent options priced only by Monte Carlo).

In [ ]:
analytical = all_greeks(params, "call")
numerical = all_fd_greeks(params, "call")

print(f"{'Greek':<8}{'Analytical':>14}{'Finite Diff':>14}{'Abs Diff':>14}")
for name in analytical:
    a, n = analytical[name], numerical[name]
    print(f"{name:<8}{a:>14.6f}{n:>14.6f}{abs(a-n):>14.2e}")

## 4. How Delta and Gamma change with the underlying price

Delta traces out an S-curve from 0 (deep OTM) to 1 (deep ITM) for a call. Gamma peaks near the strike -- this is where the option's sensitivity to the underlying changes fastest, and why at-the-money options are hardest to hedge.

In [ ]:
S_range = np.linspace(50, 150, 200)
deltas = [delta(BlackScholesParams(S=s, K=100.0, T=1.0, r=0.05, sigma=0.2), "call") for s in S_range]
gammas = [gamma(BlackScholesParams(S=s, K=100.0, T=1.0, r=0.05, sigma=0.2)) for s in S_range]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(S_range, deltas, color="steelblue")
axes[0].axvline(100, color="gray", linestyle="--", label="Strike (K=100)")
axes[0].set_xlabel("Underlying price S")
axes[0].set_ylabel("Delta")
axes[0].set_title("Call Delta vs. underlying price")
axes[0].legend()

axes[1].plot(S_range, gammas, color="steelblue")
axes[1].axvline(100, color="gray", linestyle="--", label="Strike (K=100)")
axes[1].set_xlabel("Underlying price S")
axes[1].set_ylabel("Gamma")
axes[1].set_title("Gamma peaks near the strike")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Theta decay as expiry approaches

Time decay accelerates as expiry nears, especially for at-the-money options -- the classic 'theta burn' options traders watch closely.

In [ ]:
T_range = np.linspace(0.02, 1.0, 200)
thetas = [theta(BlackScholesParams(S=100.0, K=100.0, T=t, r=0.05, sigma=0.2), "call") for t in T_range]

plt.figure(figsize=(9, 5))
plt.plot(T_range, thetas, color="steelblue")
plt.xlabel("Time to expiry (years)")
plt.ylabel("Theta (per year)")
plt.title("At-the-money call Theta accelerates as expiry approaches")
plt.gca().invert_xaxis()  # so the chart reads left-to-right as time passes
plt.show()

## Takeaways

- Monte Carlo pricing converges to the Black-Scholes closed-form price, with the classic 1/sqrt(N) error decay -- a useful sanity check whenever building a new pricing model where no closed form exists.
- Analytical and finite-difference Greeks agree closely, validating the finite-difference approach for cases (path-dependent or American options, later modules) where no closed form is available.
- Delta traces an S-curve in the underlying price; Gamma peaks at-the-money -- exactly where hedging an option position is hardest, since Delta itself is changing fastest there.
- Theta decay is not linear in time -- it accelerates as expiry approaches, especially for at-the-money options.